In [1]:
import json
import os
from pathlib import Path

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

from config import YANDEX_MODEL
from src.llm_client import analyze_style, generate_draft
from src.logger import get_recent
from src.risk_router import RiskRouter

print(f'✅ LLM подключена: {YANDEX_MODEL}')

✅ LLM подключена: yandexgpt


In [2]:
sample_messages = [
    'Да, без проблем',
    'Сейчас посмотрю',
    'Давай завтра',
    'Ок',
    'Понял, спасибо',
    'Не вопрос',
    'Я на связи',
    'Скину чуть позже',
    'Супер',
    'Напиши после 18:00',
]
profile = await analyze_style(sample_messages)
profile

{'tone': 'вежливый и лаконичный',
 'length': 'коротко',
 'emoji_usage': 'нет',
 'punctuation': 'минимальное использование пунктуации',
 'slang': 'отсутствует',
 'formality_1_10': 5,
 'common_patterns': ['краткие ответы',
  'подтверждение и согласие',
  'указание времени']}

In [3]:
incoming_batch = ['Привет', 'Слушай, вопрос есть', 'Можешь занять 5000 до среды?']
batched_text = '\n'.join(incoming_batch)
style_profile = profile if isinstance(profile, dict) else {}
result = await generate_draft(
    batched_text=batched_text,
    sender_name='Игорь',
    chat_type='private',
    history=[],
    style_profile=style_profile,
)
router = RiskRouter()
print('batched_text =')
print(batched_text)
print('result =')
print(json.dumps(result, ensure_ascii=False, indent=2))

batched_text =
Привет
Слушай, вопрос есть
Можешь занять 5000 до среды?
result =
{
  "draft": "Привет, давай обсудим это завтра, сейчас нет возможности.",
  "risk_level": "MEDIUM",
  "risk_reason": "Запрос на займ денег",
  "should_autosend": false,
  "tone_used": "нейтральный",
  "context_summary": "Игорь просит занять 5000 до среды"
}


In [4]:
test_cases = [
    {'sender': 'Максим', 'messages': ['Когда встреча завтра?']},
    {'sender': 'Оля', 'messages': ['Кинешь ссылку на отчет?', 'Нужно до вечера']},
    {'sender': 'Игорь', 'messages': ['Привет', 'Слушай', 'Можешь занять 5000 до пятницы?']},
    {'sender': 'Анна', 'messages': ['Куда отправить документы?', 'И когда созвон?']},
    {'sender': 'Неизвестный', 'messages': ['Привет', 'Ты кто вообще?']},
]
rows = []
for case in test_cases:
    batched_text = '\n'.join(case['messages'])
    generated = {
        'draft': 'Черновик для демонстрации',
        'risk_level': 'LOW' if len(case['messages']) == 1 else 'MEDIUM',
        'should_autosend': len(case['messages']) == 1,
    }
    action = 'AUTO_SEND' if generated['risk_level'] == 'LOW' and generated['should_autosend'] else 'DRAFT'
    rows.append({
        'отправитель': case['sender'],
        'пачка сообщений': batched_text,
        'черновик': generated['draft'],
        'риск': generated['risk_level'],
        'действие роутера': action,
    })
pd.DataFrame(rows)

,отправитель,пачка сообщений,черновик,риск,действие роутера
0,Максим,Когда встреча завтра?,Черновик для демонстрации,LOW,AUTO_SEND
1,Оля,Кинешь ссылку на отчет?\nНужно до вечера,Черновик для демонстрации,MEDIUM,DRAFT
2,Игорь,Привет\nСлушай\nМожешь занять 5000 до пятницы?,Черновик для демонстрации,MEDIUM,DRAFT
3,Анна,Куда отправить документы?\nИ когда созвон?,Черновик для демонстрации,MEDIUM,DRAFT
4,Неизвестный,Привет\nТы кто вообще?,Черновик для демонстрации,MEDIUM,DRAFT


In [5]:
try:
    recent = await get_recent(10)
    df = pd.DataFrame(recent)
    df
except Exception as exc:
    print(f'Не удалось прочитать базу: {exc}')

In [6]:
print("""
[Telegram]
    │  incoming message
    ▼
[bot.py — aiogram polling]
    │  buffer.add(chat_id, text)
    ▼
[MessageBuffer]  ← ждёт 20 сек после последнего сообщения
    │  on_buffer_ready(chat_id, ["msg1","msg2","msg3"])
    ▼
[llm_client.generate_draft()]  ← Yandex AI Studio API
    │  {draft, risk_level, should_autosend, ...}
    ▼
[RiskRouter]
    ├─ LOW  → risk_check() → AUTO_SEND → [Собеседник получает ответ]
    ├─ MED  → DRAFT → [Владелец: ✅ Отправить / ✏️ Редактировать / ❌ Отклонить]
    └─ HIGH → BLOCK → [Владелец: уведомление об опасном сообщении]
    │
    ▼
[logger.py → SQLite: assistant.db]
""")


[Telegram]
    │  incoming message
    ▼
[bot.py — aiogram polling]
    │  buffer.add(chat_id, text)
    ▼
[MessageBuffer]  ← ждёт 20 сек после последнего сообщения
    │  on_buffer_ready(chat_id, ["msg1","msg2","msg3"])
    ▼
[llm_client.generate_draft()]  ← Yandex AI Studio API
    │  {draft, risk_level, should_autosend, ...}
    ▼
[RiskRouter]
    ├─ LOW  → risk_check() → AUTO_SEND → [Собеседник получает ответ]
    ├─ MED  → DRAFT → [Владелец: ✅ Отправить / ✏️ Редактировать / ❌ Отклонить]
    └─ HIGH → BLOCK → [Владелец: уведомление об опасном сообщении]
    │
    ▼
[logger.py → SQLite: assistant.db]

